# Medicare Provider Analytics: SQL-Driven EDA

This notebook performs exploratory data analysis by querying PostgreSQL directly (no CSV loading).

## Scope
- Yearly spending trends
- Provider distributions
- Specialty distributions
- State-level spending
- HCPCS procedure trends
- Insight documentation

In [ ]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5433')
DB_NAME = os.getenv('DB_NAME', 'medicare_provider_analytics')
DB_USER = os.getenv('DB_USER', 'medicare_user')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'medicare_password')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

def run_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

print(f'Connected target: {DB_HOST}:{DB_PORT}/{DB_NAME}')

In [ ]:
validation_sql = '''
SELECT 'dim_provider' AS table_name, COUNT(*) AS row_count FROM dim_provider
UNION ALL
SELECT 'dim_geography' AS table_name, COUNT(*) AS row_count FROM dim_geography
UNION ALL
SELECT 'dim_service' AS table_name, COUNT(*) AS row_count FROM dim_service
UNION ALL
SELECT 'fact_provider_service' AS table_name, COUNT(*) AS row_count FROM fact_provider_service
UNION ALL
SELECT 'years_loaded' AS table_name, COUNT(DISTINCT data_year) AS row_count FROM fact_provider_service
ORDER BY table_name;
'''

df_validation = run_sql(validation_sql)
df_validation

## 1) Yearly Spending Trends

In [ ]:
yearly_spending_sql = '''
SELECT
    f.data_year,
    SUM(f.tot_srvcs) AS total_services,
    SUM(f.avg_sbmtd_chrg * f.tot_srvcs) AS total_submitted_charges,
    SUM(f.avg_mdcr_alowd_amt * f.tot_srvcs) AS total_allowed_amount,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_medicare_payment
FROM fact_provider_service f
GROUP BY f.data_year
ORDER BY f.data_year;
'''

df_yearly = run_sql(yearly_spending_sql)
df_yearly['payment_yoy_pct'] = df_yearly['total_medicare_payment'].pct_change() * 100
df_yearly

if HAS_MPL:
    ax = df_yearly.plot(x='data_year', y='total_medicare_payment', kind='line', marker='o', figsize=(10, 4), title='Total Medicare Payment by Year')
    ax.set_xlabel('Year')
    ax.set_ylabel('Payment ($)')
    plt.show()

## 2) Provider Distributions

In [ ]:
provider_dist_sql = '''
SELECT
    p.provider_type,
    COUNT(DISTINCT p.provider_key) AS provider_count
FROM dim_provider p
GROUP BY p.provider_type
ORDER BY provider_count DESC;
'''

top_providers_sql = '''
SELECT
    p.npi,
    p.last_org_name,
    p.provider_type,
    SUM(f.tot_srvcs) AS total_services,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_payment
FROM fact_provider_service f
JOIN dim_provider p ON f.provider_key = p.provider_key
GROUP BY p.npi, p.last_org_name, p.provider_type
ORDER BY total_payment DESC
LIMIT 20;
'''

df_provider_dist = run_sql(provider_dist_sql)
df_top_providers = run_sql(top_providers_sql)

print('Provider Type Distribution (Top 15):')
display(df_provider_dist.head(15))
print('Top 20 Providers by Total Payment:')
display(df_top_providers)

## 3) Specialty Distributions

In [ ]:
specialty_sql = '''
SELECT
    p.provider_type AS specialty,
    COUNT(DISTINCT p.provider_key) AS provider_count,
    SUM(f.tot_srvcs) AS total_services,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_payment
FROM fact_provider_service f
JOIN dim_provider p ON f.provider_key = p.provider_key
GROUP BY p.provider_type
ORDER BY total_payment DESC;
'''

df_specialty = run_sql(specialty_sql)
df_specialty['payment_share_pct'] = (df_specialty['total_payment'] / df_specialty['total_payment'].sum()) * 100
df_specialty.head(20)

## 4) State-Level Spending

In [ ]:
state_spending_sql = '''
SELECT
    g.state_abbreviation,
    f.data_year,
    COUNT(DISTINCT f.provider_key) AS provider_count,
    SUM(f.tot_srvcs) AS total_services,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_payment
FROM fact_provider_service f
JOIN dim_geography g ON f.geography_key = g.geography_key
GROUP BY g.state_abbreviation, f.data_year
ORDER BY f.data_year, total_payment DESC;
'''

latest_year_top_states_sql = '''
WITH latest_year AS (
    SELECT MAX(data_year) AS yr FROM fact_provider_service
)
SELECT
    g.state_abbreviation,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_payment
FROM fact_provider_service f
JOIN dim_geography g ON f.geography_key = g.geography_key
JOIN latest_year ly ON f.data_year = ly.yr
GROUP BY g.state_abbreviation
ORDER BY total_payment DESC
LIMIT 15;
'''

df_state = run_sql(state_spending_sql)
df_top_states_latest = run_sql(latest_year_top_states_sql)

print('Latest Year Top 15 States by Payment:')
display(df_top_states_latest)

if HAS_MPL:
    pivot_state = df_state.pivot_table(index='data_year', columns='state_abbreviation', values='total_payment', aggfunc='sum')
    top_states = df_top_states_latest['state_abbreviation'].tolist()[:8]
    ax = pivot_state[top_states].plot(figsize=(11, 5), title='State Payment Trends (Top States)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Payment ($)')
    plt.show()

## 5) HCPCS Procedure Trends

In [ ]:
hcpcs_trends_sql = '''
SELECT
    s.hcpcs_code,
    s.hcpcs_description,
    f.data_year,
    SUM(f.tot_srvcs) AS total_services,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_payment
FROM fact_provider_service f
JOIN dim_service s ON f.service_key = s.service_key
GROUP BY s.hcpcs_code, s.hcpcs_description, f.data_year
HAVING SUM(f.tot_srvcs) > 1000
ORDER BY f.data_year, total_payment DESC;
'''

hcpcs_latest_year_sql = '''
WITH latest_year AS (
    SELECT MAX(data_year) AS yr FROM fact_provider_service
)
SELECT
    s.hcpcs_code,
    s.hcpcs_description,
    SUM(f.tot_srvcs) AS total_services,
    SUM(f.avg_mdcr_pymt_amt * f.tot_srvcs) AS total_payment
FROM fact_provider_service f
JOIN dim_service s ON f.service_key = s.service_key
JOIN latest_year ly ON f.data_year = ly.yr
GROUP BY s.hcpcs_code, s.hcpcs_description
ORDER BY total_payment DESC
LIMIT 20;
'''

df_hcpcs_trends = run_sql(hcpcs_trends_sql)
df_hcpcs_latest = run_sql(hcpcs_latest_year_sql)

print('Top 20 HCPCS Procedures by Latest-Year Payment:')
display(df_hcpcs_latest)

## 6) Documented Insights

Run the next cell to auto-generate a concise insight summary from query outputs.

In [ ]:
latest_year = int(df_yearly['data_year'].max())
latest_payment = float(df_yearly.loc[df_yearly['data_year'] == latest_year, 'total_medicare_payment'].iloc[0])
first_year = int(df_yearly['data_year'].min())
first_payment = float(df_yearly.loc[df_yearly['data_year'] == first_year, 'total_medicare_payment'].iloc[0])
total_growth_pct = ((latest_payment - first_payment) / first_payment * 100) if first_payment else np.nan

top_provider = df_top_providers.iloc[0]
top_specialty = df_specialty.sort_values('total_payment', ascending=False).iloc[0]
top_state = df_top_states_latest.iloc[0]
top_hcpcs = df_hcpcs_latest.iloc[0]

insights = [
    f'- Yearly spending trend: Medicare payment moved from ${first_payment:,.0f} in {first_year} to ${latest_payment:,.0f} in {latest_year} ({total_growth_pct:.2f}% change).',
    f'- Provider concentration: Top provider by payment is NPI {top_provider["npi"]} ({top_provider["provider_type"]}) at ${top_provider["total_payment"]:,.0f}.',
    f'- Specialty concentration: Highest-spending specialty is {top_specialty["specialty"]} with ${top_specialty["total_payment"]:,.0f} ({top_specialty["payment_share_pct"]:.2f}% share).',
    f'- State concentration: Top latest-year state is {top_state["state_abbreviation"]} at ${top_state["total_payment"]:,.0f}.',
    f'- HCPCS trend: Top latest-year HCPCS is {top_hcpcs["hcpcs_code"]} with ${top_hcpcs["total_payment"]:,.0f} and {top_hcpcs["total_services"]:,.0f} services.'
]

print('EDA Insights Summary')
for line in insights:
    print(line)